In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="galsenai/anta_women_tts", 
    repo_type="dataset", local_dir="./anta_women_tts", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [00:09<00:00,  1.30it/s]


'/home/ubuntu/anta_women_tts'

In [3]:
files = glob('anta_women_tts/*/*.parquet')
len(files)

12

In [4]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [5]:
data = multiprocessing(files, loop, cores = 10)

100%|██████████| 1663/1663 [04:41<00:00,  5.91it/s]


In [6]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'anta_women_tts_audio/anta_women_tts-data-train-00000-of-00012_0.mp3',
 'text': 'defarkatu mburu la.',
 'speaker': 'anta_women_tts_audio'}

In [7]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'anta_women_tts')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 137.57ba/s]
Processing Files (1 / 1): 100%|██████████|  698kB /  698kB,  202kB/s  
New Data Upload: 100%|██████████|  698kB /  698kB,  202kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.74 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/c557796046407dc61e3fdb93dafc3d2976714bd4', commit_message='Upload dataset', commit_description='', oid='c557796046407dc61e3fdb93dafc3d2976714bd4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [8]:
audio_files = [d['audio_filename'] for d in data]

with open('anta_women_tts-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq anta_women_tts_audio.zip anta_women_tts_audio
# !hf upload malaysia-ai/Multilingual-TTS anta_women_tts_audio.zip --repo-type=dataset